# Notebook 1 — CV by Geography Size

The project's central empirical finding: **smaller geography = noisier estimates**. We compare **CV (coefficient of variation)** distributions across county, tract, and block group.

**CV** = standard error / estimate, where `SE = MOE / 1.645` because ACS MOEs are 90% confidence intervals. Unitless — comparable across variables.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'JL_Analysis':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'analysis' / 'JL_Analysis'
sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import GEO_LABELS, GEO_LEVELS, VARIABLES, build_cv_long

In [ ]:
cv_all = pd.concat([build_cv_long(level) for level in GEO_LEVELS], ignore_index=True)
cv_plot = cv_all.dropna(subset=['cv']).copy()
cv_plot['Geography level'] = cv_plot['geography_level'].map(GEO_LABELS)
print(f'Valid CV rows: {len(cv_plot):,}')
cv_plot.groupby(['Geography level', 'variable']).agg(
    n=('cv', 'count'), median_cv=('cv', 'median'), max_moe=('moe', 'max')
).round(3)

## CV distributions by geography size

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
groups = ['population', 'income', 'poverty', 'subgroup']
geo_order = [GEO_LABELS[l] for l in GEO_LEVELS]
for ax, group in zip(axes, groups):
    sub = cv_plot[cv_plot['variable_group'] == group]
    data = [sub.loc[sub['Geography level'] == g, 'cv'].values for g in geo_order]
    ax.boxplot(data, tick_labels=geo_order, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(group.title())
    ax.set_ylabel('CV (log scale)')
    ax.tick_params(axis='x', rotation=15)
fig.suptitle('CV distributions by geography size and variable group', y=1.02)
plt.tight_layout()
plt.show()

## Headline check — income MOE at extremes

HANDOFF.md preview: worst county income MOE ±$3,982 vs. worst block group ±$247,531.

In [ ]:
income = cv_all[cv_all['variable_code'] == 'B19013_001'].dropna(subset=['moe'])
for level in GEO_LEVELS:
    sub = income[income['geography_level'] == level]
    row = sub.loc[sub['moe'].idxmax()]
    print(f"{GEO_LABELS[level]:14s}  worst income MOE: ±${row['moe']:,.0f}  ({row['NAME']})")

county_worst = income[income['geography_level'] == 'county']['moe'].max()
bg_worst = income[income['geography_level'] == 'block_group']['moe'].max()
print(f"\nRatio (block group / county worst MOE): {bg_worst / county_worst:,.1f}x")

### Takeaway

CV and raw MOE both grow dramatically as geography shrinks — especially for income at block-group level. This is the empirical backbone of the reliability dashboard.